# EEML 2026 — Figure Generation

Interactive notebook for tweaking all figure components before Inkscape assembly.

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

from visualization import (
    apply_style, load_results,
    plot_motif_graph, plot_motif_graphs_row,
    plot_mts_heatmap, plot_spi_heatmap, plot_spi_stack,
    plot_sample_efficiency, plot_sample_efficiency_multi,
    plot_family_weights, plot_family_bar, plot_per_seed_strip,
    MODEL_COLORS, MODEL_LABELS, FAMILY_COLORS,
)

apply_style()

# Paths
DATA = Path("data/260327_eeml")
RESULTS_MAIN = Path("results/sample_efficiency_definitive_w60_cpu_results.json")
RESULTS_EDGE = Path("results/sample_efficiency_edge_ablation_cpu_results.json")
NPZ = DATA / "var-chain" / "M10_T500_I0" / "spi_mpis.npz"
TS = DATA / "var-chain" / "M10_T500_I0" / "timeseries.npy"

## Figure 1A — Motif Graphs

In [ ]:
fig, axes = plot_motif_graphs_row(M=10, figsize=(7, 2.5))
plt.show()

## Figure 1A — MTS Carpet Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(9, 2))
for ax, cls in zip(axes, ["var-chain", "var-fork", "var-collider"]):
    ts_path = DATA / cls / "M10_T500_I0" / "timeseries.npy"
    if ts_path.exists():
        plot_mts_heatmap(ts_path, ax=ax, title=cls.replace("var-", "").title())
fig.tight_layout()
plt.show()

## Figure 1B — SPI Heatmap Stack

In [ ]:
# Check available SPIs
with np.load(NPZ) as npz:
    available = sorted(npz.files)
print(f"{len(available)} SPIs available")
print("First 10:", available[:10])

In [ ]:
# Tweak which SPIs to show and their display names
spis = [
    ("xcorr_mean_sig-True", "Cross-corr $\\bar{r}$"),
    ("cov_EmpiricalCovariance", "Covariance"),
    ("mi_gaussian", "MI (Gaussian)"),
    ("sgc_parametric_mean_fs-1_fmin-0-25_fmax-0-5_order-1", "SGC (0.25\u20130.5 Hz)"),
    ("gc_gaussian_k-1_kt-1_l-1_lt-1", "GC (Gaussian)"),
    ("te_kraskov_k-4", "TE (Kraskov)"),
]
valid = [(n, t) for n, t in spis if n in available]

fig, axes = plot_spi_stack(NPZ, [v[0] for v in valid], titles=[v[1] for v in valid])
plt.show()

## Figure 1B — Markov Equivalence: Chain vs Fork

In [ ]:
# Symmetric (Pearson) looks the same for chain and fork; directed (SGC) looks different
fig, axes = plt.subplots(2, 2, figsize=(5, 5))

for col, cls in enumerate(["var-chain", "var-fork"]):
    cls_npz = DATA / cls / "M10_T500_I0" / "spi_mpis.npz"
    plot_spi_heatmap(cls_npz, "xcorr_mean_sig-True", ax=axes[0, col],
                     title=f"{cls.replace('var-', '').title()} — $|r|$ (sym)")
    plot_spi_heatmap(cls_npz, "sgc_parametric_mean_fs-1_fmin-0-25_fmax-0-5_order-1",
                     ax=axes[1, col],
                     title=f"{cls.replace('var-', '').title()} — SGC (directed)")

fig.tight_layout()
plt.show()

## Figure 1C — Family Weight Bar (compact)

In [ ]:
fig, ax = plot_family_bar(RESULTS_MAIN, n_value=500, figsize=(3.5, 2.2))
plt.show()

---
## Figure 2A — Sample Efficiency Curves

In [ ]:
fig, ax = plot_sample_efficiency(
    RESULTS_MAIN,
    models=["spi-mpnn", "fixed-spi", "mlp-mix",
            "correlation", "latent", "shuffled", "node-only"],
    figsize=(6, 4),
)
plt.show()

In [ ]:
# With edge-ablation overlay
fig, ax = plot_sample_efficiency(
    RESULTS_MAIN,
    models=["spi-mpnn", "fixed-spi", "mlp-mix",
            "correlation", "latent", "shuffled", "node-only"],
    figsize=(6, 4),
)

# Overlay edge-ablation
if RESULTS_EDGE.exists():
    edge_data = load_results(RESULTS_EDGE)
    ns_e, means_e, stds_e = [], [], []
    for n_str, v in edge_data["results"].items():
        m = v["models"].get("edge-ablation")
        if m:
            ns_e.append(int(n_str))
            means_e.append(m["f1_mean"])
            stds_e.append(m["f1_std"])
    if ns_e:
        ns_e, means_e, stds_e = zip(*sorted(zip(ns_e, means_e, stds_e)))
        means_e, stds_e = np.array(means_e), np.array(stds_e)
        ax.plot(ns_e, means_e, color=MODEL_COLORS["edge-ablation"],
                marker="D", label=MODEL_LABELS["edge-ablation"],
                linestyle="--", linewidth=1.5, markersize=4, zorder=3)
        ax.fill_between(ns_e, means_e - stds_e,
                        np.minimum(means_e + stds_e, 1.0),
                        color=MODEL_COLORS["edge-ablation"],
                        alpha=0.12, zorder=2)
        ax.legend(loc="lower right", framealpha=0.9, ncol=2)

plt.show()

## Figure 2B — Family Weights Violin

In [ ]:
fig, ax = plot_family_weights(RESULTS_MAIN, n_value=500, figsize=(4, 3))
plt.show()

## Figure 2C — Per-Seed Strip Plot

In [ ]:
fig, ax = plot_per_seed_strip(
    RESULTS_MAIN, n_value=500,
    models=["spi-mpnn", "fixed-spi", "mlp-mix", "correlation", "latent"],
    figsize=(5, 3),
)
plt.show()

## Composed Panels — Figure 2 (full)

In [ ]:
fig = plt.figure(figsize=(10, 3.5))
gs = fig.add_gridspec(1, 3, width_ratios=[2.5, 1.2, 1.3], wspace=0.35)

# Panel A
ax_a = fig.add_subplot(gs[0])
plot_sample_efficiency(
    RESULTS_MAIN,
    models=["spi-mpnn", "fixed-spi", "mlp-mix",
            "correlation", "latent", "shuffled", "node-only"],
    ax=ax_a,
)
ax_a.set_title("A", fontsize=11, fontweight="bold", loc="left")

# Panel B
ax_b = fig.add_subplot(gs[1])
plot_family_bar(RESULTS_MAIN, n_value=500, ax=ax_b)
ax_b.set_title("B", fontsize=11, fontweight="bold", loc="left")

# Panel C
ax_c = fig.add_subplot(gs[2])
plot_per_seed_strip(
    RESULTS_MAIN, n_value=500,
    models=["spi-mpnn", "fixed-spi", "correlation", "latent"],
    ax=ax_c,
)
ax_c.set_title("C", fontsize=11, fontweight="bold", loc="left")

plt.show()

## Export All
Run the CLI to regenerate all PDFs in `figures/`:

In [ ]:
!python visualization.py